In [1]:
import duckdb
import pandas as pd
import os
import json
from dotenv import load_dotenv


In [2]:
def extrair_features_moodle(pasta_dados="export/", curso_id="10464"):
    """
    Varre os logs brutos do Moodle e consolida métricas comportamentais 
    por aluno usando processamento analítico do DuckDB.
    """
    
    query_features = f"""
        SELECT 
            username,
            -- 1. Features de Volume de Engajamento
            COUNT(*) AS total_cliques,
            
            -- Converte o timestamp Unix para Data e conta os dias distintos estudados
            COUNT(DISTINCT CAST(to_timestamp(CAST(timecreated AS BIGINT)) AS DATE)) AS dias_ativos,
            
            -- 2. Features Comportamentais (Ativo vs Passivo)
            SUM(CASE WHEN crud IN ('c', 'u') THEN 1 ELSE 0 END) AS interacoes_ativas,
            SUM(CASE WHEN crud = 'r' THEN 1 ELSE 0 END) AS interacoes_passivas,
            
            -- 3. Features de Foco em Componentes
            SUM(CASE WHEN component = 'mod_forum' THEN 1 ELSE 0 END) AS cliques_forum,
            SUM(CASE WHEN component = 'mod_quiz' THEN 1 ELSE 0 END) AS cliques_quiz,
            SUM(CASE WHEN component IN ('mod_resource', 'mod_book', 'mod_page') THEN 1 ELSE 0 END) AS cliques_materiais
            
        FROM read_csv_auto('{pasta_dados}mdl_logstore_standard_log.csv', ALL_VARCHAR=TRUE)
        WHERE username NOT IN ('0', '-1', '', 'nan') 
          AND username IS NOT NULL
          AND courseid = '{curso_id}'
        GROUP BY username
        ORDER BY total_cliques DESC
    """
    
    # Executa a query e já devolve um DataFrame Pandas limpinho
    df_features = duckdb.query(query_features).df()
    return df_features

df = extrair_features_moodle()
print(df.head())

                  username  total_cliques  dias_ativos  interacoes_ativas  \
0  user6442803380426375169          31212           86             5250.0   
1  user8540069828419911681          24996           71            22080.0   
2  user7619359643386511361          17247           39             4721.0   
3  user7959886181284446209           7180           38              212.0   
4  user2959087468848087041           6685           36              679.0   

   interacoes_passivas  cliques_forum  cliques_quiz  cliques_materiais  
0              25806.0        18262.0         460.0             1376.0  
1               2892.0         1240.0          50.0              260.0  
2              12519.0        15427.0          33.0              182.0  
3               6965.0         5871.0         139.0              228.0  
4               6002.0         4658.0          37.0               77.0  


In [3]:
def extrair_notas_medias(pasta_dados="export/"):
    """
    Extrai a nota mais recente de cada item para cada aluno
    e calcula a nota média final (Target).
    """
    query_notas = f"""
        WITH NotasRecentes AS (
            -- Passo 1: Pegar apenas a nota mais recente (última modificação) de cada atividade por aluno
            SELECT 
                username,
                itemid,
                arg_max(CAST(finalgrade AS FLOAT), CAST(timemodified AS BIGINT)) as nota_recente
            FROM read_csv_auto('{pasta_dados}mdl_grade_grades_history.csv', ALL_VARCHAR=TRUE)
            WHERE finalgrade NOT IN ('', 'nan', 'None') AND finalgrade IS NOT NULL
              AND username NOT IN ('0', '-1', '', 'nan')
            GROUP BY username, itemid
        )
        -- Passo 2: Calcular a nota média por aluno a partir de suas notas mais recentes
        SELECT 
            username,
            AVG(nota_recente) AS nota_media
        FROM NotasRecentes
        GROUP BY username
        ORDER BY nota_media DESC
    """
    
    # Executa a query e retorna o DataFrame
    return duckdb.query(query_notas).df()

# Executa a função e guarda o DataFrame com o nosso "Target"
df_target = extrair_notas_medias()

# Exibe as primeiras linhas para validação
display(df_target.head())

,username,nota_media
0,user2111797327378251777,80.0
1,user968703755916673025,80.0
2,user8749729812320354305,80.0
3,user7863879879668793345,80.0
4,user3196961191700201473,80.0


In [4]:
load_dotenv()

export_path = os.getenv("PASTA_DADOS")

atividades_path = os.path.join(export_path, "atividades.json")
nivel_desordem_path = os.path.join(export_path, "nivel_desordem.json")
visualizacoes_por_objeto_path = os.path.join(export_path, "visualizacoes_por_objeto.json")
proporcao_visualizacoes_path = os.path.join(export_path, "proporcao_visualizacoes_por_atividade.json")
pontuacao_path = os.path.join(export_path, "pontuacao.json")
porcentagem_curso_path = os.path.join(export_path, "porcentagem_curso_acessada.json")
tentativas_por_questionario_path = os.path.join(export_path, "tentativas_por_questionario.json")
tempo_resposta_path = os.path.join(export_path, "tempo_resposta.json")


with open(nivel_desordem_path, 'r') as f:
    dados_json = json.load(f)

dados_por_aluno = {}

sessoes = dados_json["Course 10464"]["sessoes"]

for id_sessao, dados_sessao in sessoes.items():
    nivel_desordem = dados_sessao.get("nivel_desordem_por_usuario", {})
    
    for username, valor_desordem in nivel_desordem.items():
        # Se o aluno ainda não existe no dicionário, cria o registro dele
        if username not in dados_por_aluno:
            dados_por_aluno[username] = {"username": username}
        
        # Cria uma coluna com o nome dinâmico da sessão
        nome_coluna = f"desordem_sessao_{id_sessao}"
        dados_por_aluno[username][nome_coluna] = valor_desordem

# 4. Convertendo o dicionário para DataFrame Pandas
df_desordem = pd.DataFrame(list(dados_por_aluno.values()))

# 5. Tratamento de Nulos
# Se um aluno participou da sessão 58964 mas não da 58968, ele ficará com NaN.
# Verificar outros métodos
df_desordem = df_desordem.fillna(0.0)

# Exibe o resultado final estruturado
display(df_desordem)

,username,desordem_sessao_58964,desordem_sessao_58966,desordem_sessao_58965,desordem_sessao_58968,desordem_sessao_58967
0,user6442803380426375169,0.0,0.9359,0.8398,0.7053,0.7640
1,user7405262314024206337,0.0,0.2515,0.4106,0.7965,0.5304
2,user6215111603398901761,0.0,0.5803,0.7316,0.0000,0.0000
3,user4993780203298750465,0.0,0.0000,0.7737,0.0000,0.0000
4,user7571738205649633281,0.0,0.0000,0.0000,0.0000,0.0000
...,...,...,...,...,...,...
2136,user1261465915271151617,0.0,0.0000,0.0000,0.0000,0.0000
2137,user2678503702146318337,0.0,0.0000,0.0000,0.0000,0.0000
2138,user2368763781883887617,0.0,0.0000,0.0000,0.0000,0.0000
2139,user2961588299455528961,0.0,0.0000,0.0000,0.0000,0.0000
